## 02. Data Cleaning 

### Cleaning the NAV History dataset

In [1]:
import pandas as pd

## Loading the NAV history  dataset for cleaning
nav_history=pd.read_csv(r"D:\bluestock_mf_capstone\data\raw\02_nav_history.csv")

## Parsing the date column to datetime format
nav_history['date']=pd.to_datetime(nav_history['date'],errors='coerce')



In [2]:
# Sorting by AMFI code and date
nav_history.sort_values(by=['amfi_code',"date"],inplace=True)
print(nav_history.head())

      amfi_code       date       nav
5750     100016 2022-01-03  520.4608
5751     100016 2022-01-04  515.0971
5752     100016 2022-01-05  521.7239
5753     100016 2022-01-06  515.7880
5754     100016 2022-01-07  515.1639


In [3]:
## Removing duplicates
nav_history=nav_history.drop_duplicates()


In [4]:
# Forward filling the missing NAV values within each fund group
nav_history['nav']=nav_history.groupby('amfi_code')['nav'].ffill()


In [5]:
# Validating NAV values to be non negative (NAV>0)
invalid_nav=nav_history[nav_history['nav']<=0]
if len(invalid_nav) > 0:
    print(f"Found {len(invalid_nav)} rows with NAV <= 0")
    
    
    nav_df = nav_df[nav_df["nav"] > 0]
else:
    print("All NAV values are valid (greater than 0)")

All NAV values are valid (greater than 0)


In [6]:
# Final check for missing values and duplicated rows
print("Cleaned dataset shape:", nav_history.shape)
print("Unique AMFI codes:", nav_history["amfi_code"].nunique())
print("Missing NAV values:", nav_history["nav"].isna().sum())
print("Duplicate rows:", nav_history.duplicated().sum())

Cleaned dataset shape: (46000, 3)
Unique AMFI codes: 40
Missing NAV values: 0
Duplicate rows: 0


In [7]:
# Saving the cleaned NAV history dataset
nav_history.to_csv(r"D:\bluestock_mf_capstone\data\processed\02_nav_history_cleaned.csv",index=False)
print("Cleaned NAV history dataset saved successfully.")

Cleaned NAV history dataset saved successfully.


### Cleaning the Investor Transcations Dataset

In [8]:
#Loading and cleaning the Investor transactions dataset
inv_trans=pd.read_csv(r"D:\bluestock_mf_capstone\data\raw\08_investor_transactions.csv")

print(inv_trans.head())
print(inv_trans.info())

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [13]:
# Standardizing the date format
## Converting transaction date to datetime format
inv_trans['transaction_date']=pd.to_datetime(inv_trans['transaction_date'],errors='coerce')

#Checking Invalid dates and removing the invalid dates
inv_trans=inv_trans.dropna(subset=['transaction_date'])
print("Invalid dates:", inv_trans["transaction_date"].isna().sum())

Invalid dates: 0


In [17]:
# Standardizing the transaction_type values

#Checking the Unique values in transaction_type column
print(inv_trans["transaction_type"].unique())

inv_trans['transaction_type']=inv_trans['transaction_type'].astype(str).str.strip().str.lower()

mapping={
    "sip":"SIP",
    "systemic investment plan":"SIP",
    "lumpsum":"Lumpsum",
    "Lump sum":"Lumpsum",
    "purchased":"Lumpsum",
    "redemption":"Redemption",
    "redeem":"Redemption"
}

inv_trans['transaction_type']=inv_trans['transaction_type'].replace(mapping)
print(inv_trans["transaction_type"].sample(4))


<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str
14130    Lumpsum
6568         SIP
24715        SIP
21548        SIP
Name: transaction_type, dtype: str


In [19]:
# Validating transaction types to be within the expected set (SIP, Lumpsum, Redemption) 
valid_txn_types = ["SIP", "Lumpsum", "Redemption"]

invalid_types = inv_trans[
    ~inv_trans["transaction_type"].isin(valid_txn_types)
]

print("Invalid transaction types:", len(invalid_types))

Invalid transaction types: 0


In [21]:
# Checking for negative or zero amounts

# Converting amount to numeric and coercing errors to NaN
inv_trans["amount_inr"] = pd.to_numeric(
    inv_trans["amount_inr"],
    errors="coerce"
)
#Finding rows with invalid amount (<=0 or non-numeric)
invalid_amount=inv_trans[(inv_trans['amount_inr']<=0 )| (inv_trans["amount_inr"].isna())]
print(f"Found {len(invalid_amount)} rows with invalid amount (<=0 or non-numeric)")

# Removing the invalid amount rows
inv_trans=inv_trans[inv_trans['amount_inr']>0]


Found 0 rows with invalid amount (<=0 or non-numeric)


In [27]:
# Inspecting the unique values in KYC status column
print(inv_trans['kyc_status'].unique())

# Standardizing the KYC status values
inv_trans['kyc_status'] = inv_trans['kyc_status'].astype(str).str.strip().str.lower()

kyc_mapping = {
    "Yes": "Verified",
    "yes":"Verified",
    "Y": "Verified",
    "y":"Verified",
    "verified": "Verified",
    "Verified": "Verified",
    "Complete": "Verified",
    "No": "Pending",
    "no":"Pending",
    "N": "Pending",
    "n":"Pending",
    "pending": "Pending",
    "Pending": "Pending"
}
inv_trans['kyc_status']=inv_trans['kyc_status'].replace(kyc_mapping)

# Validating KYC status values to be within the expected set (VERIFIED, PENDING)
valid_kyc = ["Verified", "Pending"]

invalid_kyc = inv_trans[
    ~inv_trans["kyc_status"].isin(valid_kyc)
]

print("Invalid KYC values:", len(invalid_kyc))

<StringArray>
['verified', 'pending']
Length: 2, dtype: str
Invalid KYC values: 0


In [28]:
# Removing Duplicated Rows
inv_trans = inv_trans.drop_duplicates()

# Saving the cleaned investor transactions dataset
inv_trans.to_csv(r"D:\bluestock_mf_capstone\data\processed\08_investor_transactions_cleaned.csv",index=False)    
print("Cleaned investor transactions dataset saved successfully.")


Cleaned investor transactions dataset saved successfully.


In [31]:
# Data quality checks on the cleaned investor transactions dataset

print("Rows:", len(inv_trans))
print("Duplicate rows:", inv_trans.duplicated().sum())
print("Missing values:\n", inv_trans.isna().sum())
print("Invalid amounts:", (inv_trans["amount_inr"] <= 0).sum())
print("Transaction Types:\n", inv_trans["transaction_type"].value_counts())
print("KYC Status:\n", inv_trans["kyc_status"].value_counts())

Rows: 32778
Duplicate rows: 0
Missing values:
 investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
Invalid amounts: 0
Transaction Types:
 transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
KYC Status:
 kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


### Cleaning the Scheme Performance Dataset

In [33]:
##Loading the dataset
performance=pd.read_csv(r"D:\bluestock_mf_capstone\data\raw\07_scheme_performance.csv")

print(performance.head(5))
print(performance.info())

   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0              11.49

In [35]:
#Identifying Return columns
return_cols = [col for col in performance.columns if "return" in col.lower()]

print(return_cols)

['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']


In [37]:
#Converting return columns to numeric and coercing errors to NaN
for col in return_cols:
    performance[col] = pd.to_numeric(performance[col], errors="coerce")


# checking invalid return values (non-numeric or negative returns)
invalid_returns=performance[return_cols].isna().sum()

print("Invalid Return Values:")
print(invalid_returns)

#Checking for rows containig invalid return values
invalid_return_rows=performance[performance[return_cols].isna().any(axis=1)]
print(f"Found {len(invalid_return_rows)} rows with invalid return values.")

Invalid Return Values:
return_1yr_pct    0
return_3yr_pct    0
return_5yr_pct    0
dtype: int64
Found 0 rows with invalid return values.


In [40]:
##FLAGING  Return Values
## A reasonable range for mutual fund returns is: -100% <= return <= 200% 
anomalies=pd.DataFrame()
for col in return_cols:
    performance[f"{col}_flag"] = performance[col].apply(
        lambda x: "Invalid" if (x < -100) or (x > 200) else "Valid"
    )
    
    anomalies = pd.concat([anomalies, performance[performance[f"{col}_flag"] == "Invalid"]], axis=0)
    
print("Anomalies:\n", anomalies)


# Alternative method to flag anomalies 
for col in return_cols:

    Q1 = performance[col].quantile(0.25)
    Q3 = performance[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = performance[
        (performance[col] < lower) |
        (performance[col] > upper)
    ]

    print(f"{col}: {len(outliers)} outliers")

Anomalies:
 Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade, return_1yr_pct_flag, return_3yr_pct_flag, return_5yr_pct_flag]
Index: []

[0 rows x 22 columns]
return_1yr_pct: 3 outliers
return_3yr_pct: 7 outliers
return_5yr_pct: 0 outliers


In [47]:
## Validating Expense Ratio values

#Converting expense ratio to numeric and coercing errors to NaN
performance["expense_ratio_pct"] = pd.to_numeric(
    performance["expense_ratio_pct"],
    errors="coerce"
)

#Checking missing values flagging invalid expense ratio values(expected range 0.1% to 2.5%)
invalid_expense=performance["expense_ratio_pct"].isna().sum()

invalid_expense=performance[(performance['expense_ratio_pct'] < 0.1) |
    (performance['expense_ratio_pct'] > 2.5)]
print("Invalid expense ratios:", len(invalid_expense))

#Viewing the invalid expense ratio rows
print(
    invalid_expense[
        ["scheme_name", "expense_ratio_pct"]
    ]
)

Invalid expense ratios: 0
Empty DataFrame
Columns: [scheme_name, expense_ratio_pct]
Index: []


In [46]:
#Removing Duplicates
performance=performance.drop_duplicates()

In [48]:
# Data quality report for the cleaned scheme performance dataset

print("\n== DATA QUALITY REPORT ==")

print("Rows:", len(performance))

print("\nMissing Values:")
print(performance.isna().sum())

print("\nExpense Ratio Issues:")
print(len(invalid_expense))

for col in return_cols:
    print(
        f"{col}: "
        f"min={performance[col].min():.2f}, "
        f"max={performance[col].max():.2f}"
    )


== DATA QUALITY REPORT ==
Rows: 40

Missing Values:
amfi_code              0
scheme_name            0
fund_house             0
category               0
plan                   0
return_1yr_pct         0
return_3yr_pct         0
return_5yr_pct         0
benchmark_3yr_pct      0
alpha                  0
beta                   0
sharpe_ratio           0
sortino_ratio          0
std_dev_ann_pct        0
max_drawdown_pct       0
aum_crore              0
expense_ratio_pct      0
morningstar_rating     0
risk_grade             0
return_1yr_pct_flag    0
return_3yr_pct_flag    0
return_5yr_pct_flag    0
dtype: int64

Expense Ratio Issues:
0
return_1yr_pct: min=4.26, max=24.93
return_3yr_pct: min=5.14, max=23.39
return_5yr_pct: min=5.43, max=23.80


In [50]:
## Saving the cleaned scheme performance dataset
performance.to_csv(r"D:\bluestock_mf_capstone\data\processed\07_scheme_performance_cleaned.csv",index=False)
print("Cleaned scheme performance dataset saved successfully.")

Cleaned scheme performance dataset saved successfully.
